# SDO Mission Analysis: Implementation
# SDO 미션 분석: 구현

**Paper**: Pesnell, W.D., Thompson, B.J., Chamberlin, P.C. (2012). "The Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 3–15. [DOI: 10.1007/s11207-011-9841-3]

---

이 노트북은 SDO 미션의 주요 특성을 정량적으로 분석하고 시각화합니다.
SOHO와의 비교, 궤도 트레이드오프, 데이터 볼륨, 과학 질문, 그리고 태양 미션 지형도를 다룹니다.

This notebook quantitatively analyzes and visualizes the key characteristics of the SDO mission.
It covers the SOHO comparison, orbit trade-offs, data volume, science questions, and the solar mission landscape.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

# Set default plot style for consistent, publication-quality figures.
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
})

---
## Part 1: SOHO vs SDO Mission Comparison / SOHO vs SDO 미션 비교

SDO는 SOHO의 후속 미션으로, 태양 관측 능력에서 획기적인 도약을 이루었습니다.
이 섹션에서는 두 미션의 주요 파라미터를 정량적으로 비교합니다.

SDO represents a revolutionary leap over SOHO in solar observing capability.
This section provides a quantitative side-by-side comparison of key mission parameters.

In [ ]:
# ── SOHO vs SDO: Key parameter comparison table ──

comparison = {
    'Parameter': [
        'Launch Year', 'Orbit', 'Distance (km)',
        'Instruments', 'EUV Imager Resolution (px)',
        'EUV Spatial Res (arcsec)', 'EUV Channels',
        'EUV Cadence (s)', 'Magnetograph Res (px)',
        'Mag Spatial Res (arcsec)', 'Mag Type',
        'Mag Cadence (s)', 'Data Rate (bit/s)',
        'Daily Data (bytes)', 'Mass (kg)', 'Power (W)',
    ],
    'SOHO': [
        1995, 'L1', 1.5e6,
        12, 1024,
        2.6, 4,
        720, 1024,
        4.0, 'LOS',
        3600, 40e3,
        3.5e9 / 8, 1861, 1400,
    ],
    'SDO': [
        2010, 'GEO', 3.6e4,
        3, 4096,
        0.6, 10,
        12, 4096,
        1.0, 'Vector',
        45, 150e6,
        1.5e12, 3000, 1500,
    ],
}

# Print formatted comparison table.
print(f"{'Parameter':<30} {'SOHO':>15} {'SDO':>15} {'Ratio':>10}")
print('=' * 72)
for i, param in enumerate(comparison['Parameter']):
    soho_val = comparison['SOHO'][i]
    sdo_val = comparison['SDO'][i]
    if isinstance(soho_val, (int, float)) and isinstance(sdo_val, (int, float)) and soho_val > 0:
        ratio = sdo_val / soho_val
        print(f"{param:<30} {str(soho_val):>15} {str(sdo_val):>15} {ratio:>10.1f}x")
    else:
        print(f"{param:<30} {str(soho_val):>15} {str(sdo_val):>15} {'—':>10}")

In [ ]:
# ── EUV Information Rate Calculation ──
# The paper claims ~22,000x improvement in information rate.
# Information rate = pixels² × channels × (1/cadence)

eit_info = 1024**2 * 4 * (1 / 720)    # SOHO/EIT
aia_info = 4096**2 * 10 * (1 / 12)     # SDO/AIA (using 10 EUV channels)

info_ratio = aia_info / eit_info
print(f"EIT information rate:  {eit_info:.1f} px²·ch/s")
print(f"AIA information rate:  {aia_info:.1f} px²·ch/s")
print(f"AIA/EIT ratio:        {info_ratio:,.0f}x")
print(f"Paper claims:         ~22,000x")

In [ ]:
# ── Bar Chart: Key Quantitative Comparisons ──

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('SOHO vs SDO: Quantitative Comparison', fontsize=15, fontweight='bold')

bar_data = [
    ('EUV Resolution (pixels)', [1024, 4096], '4x'),
    ('EUV Channels', [4, 10], '2.5x'),
    ('EUV Cadence (s)', [720, 12], '60x faster'),
    ('Data Rate (Mbit/s)', [0.04, 150], '3,750x'),
    ('Daily Data Volume (TB)', [3.5e9 / 8 / 1e12, 1.5], '~3,500x'),
    ('Number of Instruments', [12, 3], 'Focused'),
]

colors = ['#3498db', '#e74c3c']
labels = ['SOHO', 'SDO']

for ax, (title, values, improvement) in zip(axes.flat, bar_data):
    bars = ax.bar(labels, values, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=10, fontweight='bold')

    # Use log scale for data spanning many orders of magnitude.
    if max(values) / max(min(values), 1e-15) > 100:
        ax.set_yscale('log')

    # Annotate improvement factor.
    ax.text(0.5, 0.92, improvement, transform=ax.transAxes,
            ha='center', fontsize=9, fontstyle='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

    # Add value labels on bars.
    for bar, val in zip(bars, values):
        label = f"{val:g}" if val >= 1 else f"{val:.4f}"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                label, ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Spatial & Temporal Resolution Scatter Plot ──

fig, ax = plt.subplots(figsize=(8, 6))

# (spatial_resolution_arcsec, cadence_seconds, label)
instruments = [
    (2.6, 720, 'EIT\n(SOHO)', '#3498db', 200),
    (0.6, 12, 'AIA\n(SDO)', '#e74c3c', 300),
    (4.0, 3600, 'MDI\n(SOHO)', '#2ecc71', 200),
    (1.0, 45, 'HMI\n(SDO)', '#f39c12', 300),
]

for spat, cad, label, color, size in instruments:
    ax.scatter(spat, cad, s=size, c=color, edgecolors='black',
              linewidth=1, zorder=5, alpha=0.85)
    ax.annotate(label, (spat, cad), textcoords='offset points',
                xytext=(15, 10), fontsize=9, fontweight='bold')

# Draw an arrow showing the direction of improvement.
ax.annotate('', xy=(0.4, 5), xytext=(3.5, 2500),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--'))
ax.text(1.5, 200, 'Improvement\ndirection', fontsize=9, color='gray',
        ha='center', fontstyle='italic', rotation=-45)

ax.set_xlabel('Spatial Resolution (arcsec)')
ax.set_ylabel('Cadence (seconds)')
ax.set_title('SOHO vs SDO: Resolution & Cadence', fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.3, 10)
ax.set_ylim(5, 5000)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 분석 결과 / Analysis Results

SDO는 SOHO 대비 거의 모든 관측 파라미터에서 획기적인 향상을 달성했습니다:
- **EUV 정보율**: ~22,000배 증가 (해상도 4배 × 채널 수 2.5배 × 시간 해상도 60배)
- **데이터 전송률**: 3,750배 증가 (40 kbit/s → 150 Mbit/s)
- **자기장 측정**: LOS에서 벡터 자기장으로 업그레이드, 해상도 4배 향상

SDO achieved revolutionary improvements over SOHO in nearly all observing parameters:
- **EUV information rate**: ~22,000x increase (4x resolution × 2.5x channels × 60x temporal)
- **Data rate**: 3,750x increase (40 kbit/s → 150 Mbit/s)
- **Magnetography**: Upgraded from LOS to vector field, 4x resolution improvement

---
## Part 2: GEO vs L1 Orbit Trade-off / GEO vs L1 궤도 트레이드오프

SDO가 L1 대신 GEO 궤도를 선택한 것은 미션 설계의 핵심 결정이었습니다.
가까운 거리에서 얻는 높은 데이터 전송률의 이점이 일식 시즌의 단점을 크게 상회했습니다.

SDO's choice of GEO over L1 was a defining mission design decision.
The high data rate advantage from the closer distance far outweighed the eclipse season penalty.

In [ ]:
# ── (a) Data Rate vs Distance: Signal strength scales as 1/r² ──

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Define distance range from LEO to L1.
distances_km = np.logspace(2, 6.5, 500)  # 100 km to ~3e6 km

# Normalize signal strength relative to GEO distance.
geo_dist = 36_000  # km
signal_relative = (geo_dist / distances_km) ** 2

# Scale data rate: SDO gets 150 Mbit/s at GEO.
data_rate_mbps = 150 * signal_relative

ax1.loglog(distances_km, data_rate_mbps, 'b-', linewidth=2)

# Mark key orbit positions.
orbits = {
    'LEO': (400, 'green'),
    'GEO (SDO)': (36_000, 'red'),
    'L1 (SOHO)': (1.5e6, 'blue'),
}
for name, (dist, color) in orbits.items():
    rate = 150 * (geo_dist / dist) ** 2
    ax1.scatter(dist, rate, s=100, c=color, zorder=5, edgecolors='black')
    ax1.annotate(f"{name}\n{rate:.1f} Mbit/s", (dist, rate),
                 textcoords='offset points', xytext=(10, 10),
                 fontsize=9, fontweight='bold',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax1.set_xlabel('Distance from Earth (km)')
ax1.set_ylabel('Achievable Data Rate (Mbit/s)')
ax1.set_title('Data Rate vs Distance (1/r² scaling)', fontweight='bold')
ax1.grid(True, alpha=0.3, which='both')
ax1.set_ylim(1e-3, 1e8)

# ── GEO vs L1 signal advantage calculation ──
l1_dist = 1.5e6  # km
signal_advantage = (l1_dist / geo_dist) ** 2
print(f"Signal strength advantage at GEO vs L1: {signal_advantage:,.0f}x")
print(f"  L1 distance:  {l1_dist:,.0f} km")
print(f"  GEO distance: {geo_dist:,.0f} km")
print(f"  Ratio squared: ({l1_dist/geo_dist:.1f})² = {signal_advantage:,.0f}")

# ── Right panel: Communication latency comparison ──
c = 3e5  # speed of light, km/s
latencies = {
    'GEO': geo_dist / c,
    'L1': l1_dist / c,
}
bars = ax2.barh(list(latencies.keys()), list(latencies.values()),
                color=['#e74c3c', '#3498db'], edgecolor='black', height=0.4)
for bar, val in zip(bars, latencies.values()):
    ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
             f"{val:.2f} s", va='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('One-way Light Travel Time (seconds)')
ax2.set_title('Communication Latency Comparison', fontweight='bold')
ax2.set_xlim(0, 6)

plt.tight_layout()
plt.show()

In [ ]:
# ── (b) GEO Eclipse Season Simulation ──
# SDO in inclined GEO (28.5°) experiences eclipses near equinoxes.
# Eclipse occurs when Earth's shadow cone intersects the satellite.
# Approximate: eclipses within ~23 days of equinox, max duration ~72 min.

# Day of year (1-365)
days = np.arange(1, 366)

# Equinox days: March ~80, September ~266
equinox_spring = 80
equinox_fall = 266

# Eclipse window: approximately ±23 days around each equinox.
eclipse_window = 23  # days
max_eclipse_min = 72  # maximum eclipse duration in minutes


def eclipse_duration(day: int) -> float:
    """Estimate eclipse duration for a given day of year.

    Uses a cosine model peaking at equinoxes with a half-width of
    eclipse_window days.

    Args:
        day: Day of year (1-365).

    Returns:
        Eclipse duration in minutes (0 if outside eclipse season).
    """
    dist_spring = abs(day - equinox_spring)
    dist_fall = abs(day - equinox_fall)
    dist = min(dist_spring, dist_fall)
    if dist > eclipse_window:
        return 0.0
    return max_eclipse_min * np.cos(np.pi / 2 * dist / eclipse_window)


eclipse_durations = np.array([eclipse_duration(d) for d in days])

# Calculate duty cycle.
total_minutes_year = 365 * 24 * 60
total_eclipse_minutes = np.sum(eclipse_durations)
duty_cycle = 1 - total_eclipse_minutes / total_minutes_year

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[2, 1])

# Top: eclipse duration across the year.
ax1.fill_between(days, eclipse_durations, alpha=0.4, color='navy')
ax1.plot(days, eclipse_durations, color='navy', linewidth=1)
ax1.axvline(equinox_spring, color='green', ls='--', alpha=0.7, label='Vernal Equinox')
ax1.axvline(equinox_fall, color='orange', ls='--', alpha=0.7, label='Autumnal Equinox')
ax1.set_ylabel('Eclipse Duration (min)')
ax1.set_title(f'SDO GEO Eclipse Seasons (duty cycle: {duty_cycle:.1%})', fontweight='bold')
ax1.legend(loc='upper right')
ax1.set_xlim(1, 365)
ax1.set_ylim(0, 80)

# Month labels.
month_starts = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax1.set_xticks(month_starts)
ax1.set_xticklabels(month_names)

# Bottom: daily observing time percentage.
observing_pct = 100 * (1 - eclipse_durations / (24 * 60))
ax2.fill_between(days, observing_pct, 100, alpha=0.3, color='red', label='Eclipse')
ax2.fill_between(days, 0, observing_pct, alpha=0.2, color='green', label='Observing')
ax2.axhline(100, color='blue', ls=':', alpha=0.5, label='L1 (100% duty cycle)')
ax2.set_ylabel('Daily Observing (%)')
ax2.set_xlabel('Day of Year')
ax2.set_xlim(1, 365)
ax2.set_ylim(94, 100.5)
ax2.set_xticks(month_starts)
ax2.set_xticklabels(month_names)
ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nTotal eclipse time per year: {total_eclipse_minutes:.0f} min ({total_eclipse_minutes/60:.1f} hours)")
print(f"Annual duty cycle: {duty_cycle:.2%}")
print(f"Eclipse-free days per year: {np.sum(eclipse_durations == 0)}")
print(f"Eclipse days per year: {np.sum(eclipse_durations > 0)}")

### 궤도 트레이드오프 분석 / Orbit Trade-off Analysis

**GEO의 장점 / GEO Advantages:**
- 신호 강도 ~1,736배 우위 → 150 Mbit/s Ka-band 전용 링크 가능
- 통신 지연 0.12초 (L1: 5초) → 실시간 명령 가능
- 전용 지상국 (White Sands) → DSN 공유 불필요

- Signal strength ~1,736x advantage → enables 150 Mbit/s dedicated Ka-band link
- Communication latency 0.12s (L1: 5s) → enables real-time commanding
- Dedicated ground station (White Sands) → no DSN sharing needed

**GEO의 단점 / GEO Disadvantages:**
- 춘/추분 전후 ~23일 일식 시즌 (최대 72분/일)
- 연간 관측 효율 ~95% (L1: 100%)
- 지구 자기권 통과에 따른 방사선 환경

- ~23-day eclipse seasons around equinoxes (up to 72 min/day)
- Annual duty cycle ~95% (L1: 100%)
- Radiation environment from Earth's magnetosphere

---
## Part 3: SDO Data Volume Analysis / SDO 데이터 볼륨 분석

SDO는 하루 ~1.5 TB의 데이터를 생산하며, 이는 당시 천문학 미션 중 최대 규모였습니다.
이 빅데이터 측면이 SDO를 독특하게 만드는 핵심 요소입니다.

SDO produces ~1.5 TB/day, making it the largest data-producing astronomy mission at the time.
This big-data aspect is a defining characteristic of the SDO mission.

In [ ]:
# ── (a) Daily Data Breakdown by Instrument ──

# AIA: 4096² pixels × 16 bits × 8 primary channels × (86400/12) images/day
aia_pixels = 4096**2
aia_bits = 16
aia_channels = 8  # primary EUV channels for data volume calc
aia_cadence_s = 12
aia_images_per_day = 86400 / aia_cadence_s
aia_bytes_per_day = aia_pixels * aia_bits / 8 * aia_channels * aia_images_per_day
aia_tb_per_day = aia_bytes_per_day / 1e12

# HMI: 4096² pixels × 16 bits × 6 observables × (86400/45) images/day
hmi_pixels = 4096**2
hmi_bits = 16
hmi_observables = 6
hmi_cadence_s = 45
hmi_images_per_day = 86400 / hmi_cadence_s
hmi_bytes_per_day = hmi_pixels * hmi_bits / 8 * hmi_observables * hmi_images_per_day
hmi_tb_per_day = hmi_bytes_per_day / 1e12

# EVE: relatively small
eve_tb_per_day = 0.001  # ~1 GB/day

total_tb_per_day = aia_tb_per_day + hmi_tb_per_day + eve_tb_per_day

print("=== SDO Daily Data Production ===")
print(f"AIA: {aia_tb_per_day:.3f} TB/day ({aia_images_per_day:.0f} images/day)")
print(f"HMI: {hmi_tb_per_day:.3f} TB/day ({hmi_images_per_day:.0f} images/day)")
print(f"EVE: {eve_tb_per_day:.3f} TB/day")
print(f"Total: {total_tb_per_day:.3f} TB/day")
print(f"\nPaper states: ~1.5 TB/day")

# Pie chart of data contribution.
fig, ax = plt.subplots(figsize=(7, 7))
sizes = [aia_tb_per_day, hmi_tb_per_day, eve_tb_per_day]
labels = [f'AIA\n{aia_tb_per_day:.2f} TB/day',
          f'HMI\n{hmi_tb_per_day:.2f} TB/day',
          f'EVE\n{eve_tb_per_day:.3f} TB/day']
colors = ['#e74c3c', '#3498db', '#2ecc71']
explode = (0.05, 0.02, 0.02)

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax.set_title(f'SDO Daily Data Volume: {total_tb_per_day:.2f} TB/day', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── (b) Cumulative Data Over Mission Lifetime ──

# SDO launched Feb 11, 2010. Simulate 15 years of data accumulation.
years = np.arange(2010, 2026)
days_per_year = 365.25

# SDO: ~1.5 TB/day
sdo_daily_tb = 1.5
sdo_cumulative_pb = np.cumsum(np.full(len(years), sdo_daily_tb * days_per_year)) / 1000

# SOHO: ~3.5 Gbit/day = ~0.4375 GB/day = ~0.0004375 TB/day
soho_daily_tb = 3.5e9 / 8 / 1e12
# SOHO started 1995, but plot from 2010 for comparison.
soho_cumulative_2010_pb = soho_daily_tb * days_per_year * (2010 - 1995) / 1000
soho_cumulative_pb = (soho_cumulative_2010_pb +
                      np.cumsum(np.full(len(years), soho_daily_tb * days_per_year)) / 1000)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(years, sdo_cumulative_pb, 'r-o', linewidth=2, markersize=5, label='SDO')
ax.plot(years, soho_cumulative_pb, 'b-s', linewidth=2, markersize=5, label='SOHO')

# Mark PB milestones for SDO.
for milestone in [1, 2, 3, 4, 5, 6, 7, 8]:
    # Find the year when cumulative data crosses the milestone.
    crossing_idx = np.searchsorted(sdo_cumulative_pb, milestone)
    if crossing_idx < len(years):
        ax.axhline(milestone, color='gray', ls=':', alpha=0.3)
        ax.text(years[0] - 0.3, milestone, f'{milestone} PB',
                fontsize=8, color='gray', va='center', ha='right')

ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Data (PB)')
ax.set_title('Cumulative Mission Data: SDO vs SOHO', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(2009.5, 2025.5)

# Add annotation for the massive difference.
ax.annotate(f'SDO: {sdo_cumulative_pb[-1]:.1f} PB by 2025',
            xy=(2025, sdo_cumulative_pb[-1]),
            xytext=(2020, sdo_cumulative_pb[-1] * 0.7),
            fontsize=10, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print(f"SDO cumulative data by 2025: {sdo_cumulative_pb[-1]:.1f} PB")
print(f"SOHO cumulative data by 2025 (since 1995): {soho_cumulative_pb[-1]:.4f} PB")
print(f"Ratio: {sdo_cumulative_pb[-1]/soho_cumulative_pb[-1]:,.0f}x")

In [ ]:
# ── (c) Data Rate Evolution Across Solar Missions ──

missions_data = {
    'Skylab\n(1973)': 0.001,           # Film-based, effectively ~kbit/s equivalent
    'SMM\n(1980)': 0.005,              # ~5 kbit/s
    'Yohkoh\n(1991)': 0.032,           # ~32 kbit/s
    'SOHO\n(1995)': 0.040,             # 40 kbit/s
    'TRACE\n(1998)': 0.150,            # ~150 kbit/s
    'Hinode\n(2006)': 2.0,             # ~2 Mbit/s
    'STEREO\n(2006)': 0.720,           # ~720 kbit/s
    'SDO\n(2010)': 150.0,              # 150 Mbit/s
}

fig, ax = plt.subplots(figsize=(12, 6))

names = list(missions_data.keys())
rates = list(missions_data.values())
colors_bar = plt.cm.plasma(np.linspace(0.2, 0.9, len(names)))

bars = ax.bar(names, rates, color=colors_bar, edgecolor='black', linewidth=0.5)
ax.set_yscale('log')
ax.set_ylabel('Data Rate (Mbit/s)')
ax.set_title('Solar Mission Data Rate Evolution', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Annotate values on bars.
for bar, rate in zip(bars, rates):
    if rate >= 1:
        label = f"{rate:.0f} Mbit/s"
    elif rate >= 0.01:
        label = f"{rate*1000:.0f} kbit/s"
    else:
        label = f"{rate*1000:.1f} kbit/s"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.3,
            label, ha='center', va='bottom', fontsize=8, fontweight='bold')

# Highlight SDO bar.
bars[-1].set_edgecolor('red')
bars[-1].set_linewidth(2)

ax.set_ylim(5e-4, 1e3)

plt.tight_layout()
plt.show()

### 데이터 볼륨 분석 결과 / Data Volume Analysis Results

SDO의 데이터 생산량은 태양 물리학에 빅데이터 시대를 열었습니다:
- **AIA**가 전체 데이터의 ~80%를 차지 (높은 해상도 + 빠른 케이던스 + 다수 채널)
- 15년간 누적 ~8 PB 이상의 데이터 생산
- SOHO 대비 일일 데이터량 ~3,500배
- 태양 미션 데이터 전송률의 지수적 성장 추세

SDO's data production ushered in the big-data era of solar physics:
- **AIA** accounts for ~80% of total data (high resolution + fast cadence + many channels)
- Over 15 years, cumulative production exceeds ~8 PB
- ~3,500x more daily data than SOHO
- Exponential growth trend in solar mission data rates

---
## Part 4: SDO Science Questions Visualization / SDO 과학 질문 시각화

SDO는 7개의 Level-1 과학 질문(Table 1)을 중심으로 설계되었습니다.
3개의 기기(AIA, HMI, EVE)가 이 질문들을 어떻게 다루는지 시각화합니다.

SDO was designed around 7 Level-1 science questions (Table 1).
We visualize how the 3 instruments (AIA, HMI, EVE) address these questions.

In [ ]:
# ── Science Question / Instrument Matrix ──

science_questions = [
    'Q1: Solar cycle\nmechanism',
    'Q2: Active region\nflux',
    'Q3: Reconnection /\ncoronal heating',
    'Q4: EUV irradiance\nvariations',
    'Q5: CME/flare magnetic\nconfigurations',
    'Q6: Solar wind\nprediction',
    'Q7: Space weather\nforecasting',
]

instruments = ['AIA', 'HMI', 'EVE']

# Contribution matrix: 2 = primary, 1 = contributing, 0 = minimal.
# Rows: questions, Columns: instruments (AIA, HMI, EVE)
contribution = np.array([
    [1, 2, 0],  # Q1: Solar cycle → HMI primary, AIA contributing
    [1, 2, 0],  # Q2: AR flux → HMI primary, AIA contributing
    [2, 1, 0],  # Q3: Reconnection → AIA primary, HMI contributing
    [1, 0, 2],  # Q4: EUV irradiance → EVE primary, AIA contributing
    [2, 1, 0],  # Q5: CME/flare → AIA primary, HMI contributing
    [1, 2, 0],  # Q6: Solar wind → HMI primary, AIA contributing
    [2, 2, 2],  # Q7: Space weather → All three
])

fig, ax = plt.subplots(figsize=(8, 8))

# Custom colormap: white (0) → light blue (1) → dark red (2).
cmap = LinearSegmentedColormap.from_list(
    'contribution', ['#f0f0f0', '#74b9ff', '#d63031'], N=3
)

im = ax.imshow(contribution, cmap=cmap, aspect='auto', vmin=0, vmax=2)

# Labels.
ax.set_xticks(range(len(instruments)))
ax.set_xticklabels(instruments, fontsize=12, fontweight='bold')
ax.set_yticks(range(len(science_questions)))
ax.set_yticklabels(science_questions, fontsize=9)
ax.xaxis.set_ticks_position('top')

# Annotate cells.
labels_map = {0: '', 1: 'Contributing', 2: 'Primary'}
for i in range(len(science_questions)):
    for j in range(len(instruments)):
        text = labels_map[contribution[i, j]]
        color = 'white' if contribution[i, j] == 2 else 'black'
        ax.text(j, i, text, ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')

ax.set_title('SDO Science Questions × Instrument Contribution Matrix',
             fontweight='bold', pad=20)

# Legend patches.
legend_elements = [
    mpatches.Patch(facecolor='#d63031', label='Primary'),
    mpatches.Patch(facecolor='#74b9ff', label='Contributing'),
    mpatches.Patch(facecolor='#f0f0f0', edgecolor='gray', label='Minimal'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Temperature / Height Coverage by Instrument ──

fig, ax = plt.subplots(figsize=(12, 5))

# AIA EUV channel temperature coverage (log T in K).
aia_channels_info = [
    ('94 A', 6.8, '#9b59b6'),
    ('131 A', 5.6, '#3498db'),
    ('171 A', 5.8, '#2ecc71'),
    ('193 A', 6.2, '#f1c40f'),
    ('211 A', 6.3, '#e67e22'),
    ('304 A', 4.7, '#e74c3c'),
    ('335 A', 6.4, '#c0392b'),
]

# Approximate temperature response width (in log T).
width = 0.4
y_aia = 2

for i, (name, log_t, color) in enumerate(aia_channels_info):
    ax.barh(y_aia, width, left=log_t - width / 2, height=0.15,
            color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.text(log_t, y_aia + 0.12, name, ha='center', fontsize=7, fontweight='bold')

# HMI: photosphere (log T ~ 3.7)
ax.barh(1, 0.3, left=3.55, height=0.15, color='#3498db',
        alpha=0.7, edgecolor='black', linewidth=0.5)
ax.text(3.7, 1.12, 'Photosphere', ha='center', fontsize=8, fontweight='bold')

# EVE: covers full EUV range (measures integrated irradiance).
ax.barh(0, 3.0, left=4.0, height=0.15, color='#2ecc71',
        alpha=0.5, edgecolor='black', linewidth=0.5)
ax.text(5.5, 0.12, 'Full EUV Irradiance', ha='center', fontsize=8, fontweight='bold')

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['EVE', 'HMI', 'AIA'], fontsize=11, fontweight='bold')
ax.set_xlabel('log T (K)')
ax.set_title('SDO Instrument Temperature Coverage', fontweight='bold')
ax.set_xlim(3.0, 7.5)
ax.set_ylim(-0.5, 2.8)
ax.grid(True, alpha=0.3, axis='x')

# Add solar region labels.
regions = [
    (3.7, 'Photosphere\n~5,800 K'),
    (4.3, 'Chromosphere\n~20,000 K'),
    (5.0, 'Transition\nRegion'),
    (6.2, 'Corona\n~1-2 MK'),
    (7.0, 'Flare\n~10 MK'),
]
for log_t, label in regions:
    ax.axvline(log_t, color='gray', ls=':', alpha=0.3)
    ax.text(log_t, -0.45, label, ha='center', fontsize=7, color='gray')

plt.tight_layout()
plt.show()

### 과학 질문 분석 / Science Question Analysis

SDO의 3개 기기는 상호보완적으로 설계되었습니다:
- **HMI**: 태양 주기 메커니즘과 자기장 관련 질문의 주요 기기 (Q1, Q2, Q6)
- **AIA**: 코로나 역학과 CME/플레어 관련 질문의 주요 기기 (Q3, Q5)
- **EVE**: EUV 복사조도 변동의 유일한 주요 기기 (Q4)
- 우주 기상 예보(Q7)는 세 기기 모두가 기여하는 통합 목표

SDO's 3 instruments are designed to be complementary:
- **HMI**: Primary instrument for solar cycle and magnetic field questions (Q1, Q2, Q6)
- **AIA**: Primary instrument for coronal dynamics and CME/flare questions (Q3, Q5)
- **EVE**: The sole primary instrument for EUV irradiance variations (Q4)
- Space weather forecasting (Q7) is an integrated goal served by all three instruments

---
## Part 5: SDO in the Solar Mission Landscape / SDO의 태양 미션 지형도

SDO는 단독이 아닌 다수의 동시대 태양 미션들과 함께 운용되었습니다.
이 섹션에서는 미션 타임라인과 역량 매트릭스를 시각화합니다.

SDO operates alongside numerous contemporaneous solar missions.
This section visualizes the mission timeline and capability matrix.

In [ ]:
# ── Solar Mission Timeline (Gantt Chart) ──

missions_timeline = [
    ('SOHO', 1995, 2026, '#3498db'),
    ('TRACE', 1998, 2010, '#95a5a6'),
    ('RHESSI', 2002, 2018, '#9b59b6'),
    ('STEREO', 2006, 2026, '#e67e22'),
    ('Hinode', 2006, 2026, '#2ecc71'),
    ('SDO', 2010, 2026, '#e74c3c'),
    ('IRIS', 2013, 2026, '#f1c40f'),
    ('Parker Solar Probe', 2018, 2026, '#1abc9c'),
    ('Solar Orbiter', 2020, 2026, '#d35400'),
]

fig, ax = plt.subplots(figsize=(14, 6))

for i, (name, start, end, color) in enumerate(missions_timeline):
    # Highlight SDO with a thicker bar.
    height = 0.7 if name == 'SDO' else 0.5
    lw = 2 if name == 'SDO' else 0.5
    alpha = 1.0 if name == 'SDO' else 0.7

    ax.barh(i, end - start, left=start, height=height,
            color=color, alpha=alpha, edgecolor='black', linewidth=lw)
    ax.text(start + 0.3, i, f"{name} ({start})",
            va='center', fontsize=9, fontweight='bold',
            color='white' if name == 'SDO' else 'black')

# SDO launch marker.
ax.axvline(2010, color='red', ls='--', alpha=0.5, linewidth=1.5)
ax.text(2010.1, len(missions_timeline) - 0.5, 'SDO Launch\nFeb 2010',
        fontsize=9, color='red', fontweight='bold')

ax.set_yticks(range(len(missions_timeline)))
ax.set_yticklabels([m[0] for m in missions_timeline], fontsize=10)
ax.set_xlabel('Year')
ax.set_title('Solar Mission Timeline', fontweight='bold', fontsize=14)
ax.set_xlim(1993, 2027)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── Mission Capability Matrix ──

capability_missions = [
    'SOHO', 'TRACE', 'STEREO', 'Hinode', 'SDO',
    'IRIS', 'PSP', 'Solar Orbiter',
]

capabilities = ['EUV/UV\nImaging', 'Spectroscopy', 'Magneto-\ngraphy',
                'In-situ', 'Coronagraphy']

# Capability matrix: 2 = strong, 1 = present, 0 = absent.
cap_matrix = np.array([
    [1, 2, 1, 1, 2],  # SOHO: EIT, SUMER/CDS/UVCS, MDI, CELIAS, LASCO
    [2, 0, 0, 0, 0],  # TRACE: EUV imaging only
    [1, 0, 0, 1, 2],  # STEREO: EUVI, SECCHI/COR, IMPACT/PLASTIC
    [1, 2, 2, 0, 0],  # Hinode: XRT/SOT, EIS, SOT/SP
    [2, 0, 2, 0, 0],  # SDO: AIA, HMI (no spectroscopy, no in-situ, no coronagraph)
    [1, 2, 0, 0, 0],  # IRIS: SJI, spectrograph
    [0, 0, 0, 2, 0],  # PSP: FIELDS, SWEAP (in-situ primary)
    [2, 1, 1, 2, 0],  # Solar Orbiter: EUI, SPICE, PHI, SWA/MAG
])

fig, ax = plt.subplots(figsize=(10, 7))

cmap_cap = LinearSegmentedColormap.from_list(
    'capability', ['#ecf0f1', '#74b9ff', '#0984e3'], N=3
)

im = ax.imshow(cap_matrix, cmap=cmap_cap, aspect='auto', vmin=0, vmax=2)

ax.set_xticks(range(len(capabilities)))
ax.set_xticklabels(capabilities, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(capability_missions)))
ax.set_yticklabels(capability_missions, fontsize=10)
ax.xaxis.set_ticks_position('top')

# Annotate and highlight SDO row.
cap_labels = {0: '-', 1: 'Yes', 2: 'Strong'}
for i in range(len(capability_missions)):
    for j in range(len(capabilities)):
        text = cap_labels[cap_matrix[i, j]]
        color = 'white' if cap_matrix[i, j] == 2 else 'black'
        fontw = 'bold' if cap_matrix[i, j] == 2 else 'normal'
        ax.text(j, i, text, ha='center', va='center',
                fontsize=9, color=color, fontweight=fontw)

# Highlight SDO row.
sdo_idx = capability_missions.index('SDO')
rect = plt.Rectangle((-0.5, sdo_idx - 0.5), len(capabilities), 1,
                      fill=False, edgecolor='red', linewidth=2.5)
ax.add_patch(rect)

ax.set_title('Solar Mission Capability Matrix', fontweight='bold', pad=20)

legend_elements = [
    mpatches.Patch(facecolor='#0984e3', label='Strong'),
    mpatches.Patch(facecolor='#74b9ff', label='Present'),
    mpatches.Patch(facecolor='#ecf0f1', edgecolor='gray', label='Absent'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

### 미션 지형도 분석 / Mission Landscape Analysis

SDO는 태양 관측 미션 중 독특한 위치를 차지합니다:
- **EUV 이미징과 자기장 측정에 특화** — 코로나그래프나 현장(in-situ) 측정 기기 없음
- SOHO의 12개 기기 접근법과 달리 3개 기기에 **집중 투자** 전략
- STEREO(다시점), Hinode(분광), IRIS(고해상도 분광)와 **상호보완적**
- Parker Solar Probe와 Solar Orbiter의 현장 측정에 대한 **원격 관측 컨텍스트** 제공

SDO occupies a unique niche among solar observing missions:
- **Specialized in EUV imaging and magnetography** — no coronagraph or in-situ instruments
- A **focused investment** strategy with 3 instruments, unlike SOHO's 12-instrument approach
- **Complementary** to STEREO (multi-viewpoint), Hinode (spectroscopy), IRIS (high-res spectroscopy)
- Provides **remote-sensing context** for Parker Solar Probe and Solar Orbiter in-situ measurements

---
## Summary / 요약

### SOHO vs SDO: Mission-Level Comparison / 미션 수준 비교

| Parameter / 파라미터 | SOHO (1995) | SDO (2010) | Significance / 의의 |
|:---|:---|:---|:---|
| Orbit / 궤도 | L1 (1.5M km) | GEO (36,000 km) | 높은 데이터 전송률 / High data rate |
| Instruments / 기기 수 | 12 | 3 (AIA+HMI+EVE) | 집중 투자 / Focused investment |
| EUV Information Rate | ~5,800 px²·ch/s | ~140M px²·ch/s | ~22,000× 증가 / increase |
| Data Rate / 전송률 | 40 kbit/s | 150 Mbit/s | 3,750× 증가 / increase |
| Daily Data / 일일 데이터 | ~0.44 GB | ~1.5 TB | ~3,500× 증가 / increase |
| Magnetography / 자기장 | LOS (MDI) | Vector (HMI) | 전벡터 자기장 / Full vector field |
| Ground Station / 지상국 | DSN (shared) | White Sands (dedicated) | 전용 링크 / Dedicated link |
| Duty Cycle / 관측 효율 | ~100% (L1) | >95% (GEO) | 일식 시즌 손실 / Eclipse penalty |
| Mission Focus / 초점 | Broad solar physics | LWS (Sun-Earth) | 우주기상 중심 / Space weather focus |

### 핵심 교훈 / Key Lessons

1. **궤도 선택은 과학을 결정한다** — GEO 선택으로 3,750배 높은 데이터율 확보, 이것이 AIA의 12초 전체 태양 이미징을 가능하게 함
2. **적은 기기, 더 나은 과학** — 12개에서 3개로 줄였지만, 각 기기가 전례 없는 성능 달성
3. **빅데이터가 새로운 과학을 연다** — 1.5 TB/day는 기계학습과 통계적 태양 물리학의 시대를 개척

1. **Orbit choice determines science** — GEO selection secured 3,750× higher data rate, enabling AIA's 12-second full-Sun imaging
2. **Fewer instruments, better science** — Reduced from 12 to 3, but each achieves unprecedented performance
3. **Big data enables new science** — 1.5 TB/day pioneered the era of machine learning and statistical solar physics

In [ ]:
# ── Final Summary: Key Numbers ──

print("=" * 60)
print("SDO Mission: Key Numbers Summary")
print("=" * 60)
print(f"EUV info rate improvement (AIA/EIT):  {info_ratio:,.0f}x")
print(f"Data rate improvement:                3,750x")
print(f"Daily data volume:                    ~1.5 TB")
print(f"Signal advantage (GEO vs L1):         {signal_advantage:,.0f}x")
print(f"GEO duty cycle:                       {duty_cycle:.1%}")
print(f"Cumulative data (15 yr):              {sdo_cumulative_pb[-1]:.1f} PB")
print(f"Communication latency (GEO):          {geo_dist/c:.2f} s")
print(f"Communication latency (L1):           {l1_dist/c:.2f} s")
print("=" * 60)

---
## References / 참고문헌

- Pesnell, W.D., Thompson, B.J., Chamberlin, P.C. (2012). "The Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 3–15. [DOI: 10.1007/s11207-011-9841-3](https://doi.org/10.1007/s11207-011-9841-3)
- Lemen, J.R. et al. (2012). "The Atmospheric Imaging Assembly (AIA) on the Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 17–40.
- Scherrer, P.H. et al. (2012). "The Helioseismic and Magnetic Imager (HMI) Investigation for the Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 207–227.
- Woods, T.N. et al. (2012). "Extreme Ultraviolet Variability Experiment (EVE) on the Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 115–143.